#### **Data Cleaning & Preparation**

**Goal:**  
Load the simulated dataset, inspect for issues, apply cleaning steps, engineer new features for analysis, and prepare a clean version for modeling and visualization.

**Input:** `../data/simulated_transactions_2023_2025.csv`  
**Output:** `../data/cleaned_transactions_2023_2025.csv`  

**Main Tasks:**
- Load & inspect
- Handle outliers / unrealistic values (even though simulation is clean)
- Create derived columns: year_quarter, price_bucket, yoy_growth_proxy, mid_range_flag, etc.
- Final validation & summary

#### **1. Imports**

In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded.")

Libraries loaded.


#### **2. Load Data & Initial Inspection**

##### **Load dataset**

In [12]:
INPUT_PATH = '../data/simulated_transactions_2023_2025.csv'
df = pd.read_csv(INPUT_PATH, parse_dates=['date'])

print(f"Loaded {len(df)} rows")

Loaded 4000 rows


##### **Data types**

In [11]:
df.dtypes

transaction_id                  int64
date                   datetime64[ns]
emirate                        object
area                           object
property_type                  object
size_sqft                       int64
price_per_sqft                float64
price_aed                     float64
buyer_type                     object
sustainability_flag              bool
is_offplan                       bool
year                            int64
dtype: object

##### **Missing values**

In [10]:
df.isna().sum()

transaction_id         0
date                   0
emirate                0
area                   0
property_type          0
size_sqft              0
price_per_sqft         0
price_aed              0
buyer_type             0
sustainability_flag    0
is_offplan             0
year                   0
dtype: int64

##### **First 5 rows**

In [9]:
df.head()

,transaction_id,date,emirate,area,property_type,size_sqft,price_per_sqft,price_aed,buyer_type,sustainability_flag,is_offplan,year
0,1,2025-06-01,Dubai,Dubai Marina,Apartment,500,2120.0,1060000.0,First-Time,True,False,2025
1,2,2023-12-24,Dubai,Downtown Dubai,Apartment,1300,2315.0,3010000.0,Investor,False,False,2023
2,3,2024-11-17,Dubai,Dubai Marina,Apartment,1240,1972.0,2445000.0,First-Time,False,False,2024
3,4,2024-02-29,Dubai,Business Bay,Apartment,1110,1844.0,2047000.0,Investor,True,True,2024
4,5,2023-05-20,Dubai,Business Bay,Apartment,740,1437.0,1063000.0,Investor,False,True,2023


##### **Descriptive stats (numeric)**

In [8]:
df.describe()

,transaction_id,date,size_sqft,price_per_sqft,price_aed,year
count,4000.000000,4000,4000.000000,4000.000000,4.000000e+03,4000.000000
mean,2000.500000,2024-06-25 04:22:26.400000256,1522.700000,1733.153250,2.643174e+06,2023.983250
min,1.000000,2023-01-01 00:00:00,500.000000,700.000000,3.880000e+05,2023.000000
25%,1000.750000,2023-09-23 00:00:00,910.000000,1214.000000,1.319750e+06,2023.000000
50%,2000.500000,2024-06-21 00:00:00,1240.000000,1518.000000,2.005500e+06,2024.000000
75%,3000.250000,2025-03-29 00:00:00,1860.000000,2188.250000,3.318500e+06,2025.000000
max,4000.000000,2025-12-30 00:00:00,5840.000000,3500.000000,1.597300e+07,2025.000000
std,1154.844867,NaN,899.471135,674.749506,1.999024e+06,0.822276


#### **3. Outlier & Range Validation**

In [15]:
# Realistic bounds (based on UAE 2023-2025 market)
MIN_SIZE_SQFT    = 400
MAX_SIZE_SQFT    = 8000
MIN_PPS          = 600
MAX_PPS          = 4000
MIN_PRICE_AED    = 300_000
MAX_PRICE_AED    = 30_000_000

# Clip extreme values
df['size_sqft'] = df['size_sqft'].clip(MIN_SIZE_SQFT, MAX_SIZE_SQFT)
df['price_per_sqft'] = df['price_per_sqft'].clip(MIN_PPS, MAX_PPS)
df['price_aed'] = df['price_aed'].clip(MIN_PRICE_AED, MAX_PRICE_AED)

# Flag any clipped rows
df['clipped'] = (
    (df['size_sqft'] == MIN_SIZE_SQFT) | (df['size_sqft'] == MAX_SIZE_SQFT) |
    (df['price_per_sqft'] == MIN_PPS) | (df['price_per_sqft'] == MAX_PPS) |
    (df['price_aed'] == MIN_PRICE_AED) | (df['price_aed'] == MAX_PRICE_AED)
)

print(f"Rows with clipping applied: {df['clipped'].sum()} ({df['clipped'].mean():.2%})")

Rows with clipping applied: 0 (0.00%)


#### **4. Feature Engineering – Derived Columns**

In [16]:
# Ensure it's int
df['year'] = df['date'].dt.year.astype(int)

# Quarter & Month
df['quarter'] = df['date'].dt.to_period('Q').astype(str)
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()

# Price buckets
bins = [0, 1_000_000, 2_000_000, 4_000_000, 7_000_000, 15_000_000, np.inf]
labels = ['<1M', '1-2M', '2-4M', '4-7M', '7-15M', '15M+']
df['price_bucket'] = pd.cut(df['price_aed'], bins=bins, labels=labels, right=False)

# Mid-range flag (target for first-time buyers)
df['mid_range'] = (df['price_aed'].between(800_000, 4_500_000)).astype(int)

# Simple proxy for YoY % change (lagged average pps per emirate/year)
pps_year_emirate = df.groupby(['emirate', 'year'])['price_per_sqft'].mean().reset_index()
pps_year_emirate['pps_prev_year'] = pps_year_emirate.groupby('emirate')['price_per_sqft'].shift(1)
pps_year_emirate['yoy_growth_pct'] = ((pps_year_emirate['price_per_sqft'] - pps_year_emirate['pps_prev_year']) 
                                      / pps_year_emirate['pps_prev_year'] * 100).round(1)

# Merge back
df = df.merge(pps_year_emirate[['emirate', 'year', 'yoy_growth_pct']], 
              on=['emirate', 'year'], how='left')

##### **New columns added**

In [20]:
df.columns.tolist()

['transaction_id',
 'date',
 'emirate',
 'area',
 'property_type',
 'size_sqft',
 'price_per_sqft',
 'price_aed',
 'buyer_type',
 'sustainability_flag',
 'is_offplan',
 'year',
 'clipped',
 'quarter',
 'month',
 'month_name',
 'price_bucket',
 'mid_range',
 'yoy_growth_pct']

##### **Sample of new features**

In [19]:
df[['date', 'year', 'quarter', 'price_bucket', 'mid_range', 'yoy_growth_pct']].sample(8)

,date,year,quarter,price_bucket,mid_range,yoy_growth_pct
1,2023-12-24,2023,2023Q4,2-4M,1,NaN
3786,2024-05-06,2024,2024Q2,7-15M,0,15.5
2443,2023-10-05,2023,2023Q4,4-7M,0,NaN
1539,2025-03-19,2025,2025Q1,2-4M,1,8.6
3202,2023-08-26,2023,2023Q3,<1M,1,NaN
607,2025-09-27,2025,2025Q3,1-2M,1,8.6
3823,2024-09-20,2024,2024Q3,2-4M,1,15.5
185,2025-05-07,2025,2025Q2,4-7M,0,6.0


#### **5. Final Validation & Save Cleaned Data** 

##### **Re-check after engineering**

In [21]:
print("\nValue counts – price_bucket:")
df['price_bucket'].value_counts(dropna=False)


Value counts – price_bucket:


price_bucket
1-2M     1521
2-4M     1325
4-7M      504
<1M       471
7-15M     177
15M+        2
Name: count, dtype: int64

In [22]:
print("\nAvg yoy_growth_pct by year & emirate:")
df.groupby(['emirate', 'year'])['yoy_growth_pct'].mean().round(1).unstack()


Avg yoy_growth_pct by year & emirate:


year,2023,2024,2025
emirate,,,
Abu Dhabi,NaN,7.8,8.6
Dubai,NaN,15.5,6.0


#### **6. Save cleaned version**

In [23]:
OUTPUT_CLEAN_PATH = '../data/cleaned_transactions_2023_2025.csv'
df.to_csv(OUTPUT_CLEAN_PATH, index=False)
print(f"\nCleaned dataset saved to: {OUTPUT_CLEAN_PATH}")


Cleaned dataset saved to: ../data/cleaned_transactions_2023_2025.csv
